# Reading the Trail: Colab runner

This notebook just drives the code in `src/`; it holds no experiment logic. Set the runtime to GPU (T4) first.

Re-running the train cell after a timeout resumes from `last.pt` on Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
REPO_URL = 'https://github.com/cnash03/Deep-Learning-Project.git'
BRANCH = 'main'
PROJECT = '/content/drive/MyDrive/reading-the-trail'  # datasets + runs live here

!git clone -b $BRANCH $REPO_URL /content/repo 2>/dev/null || (cd /content/repo && git pull)
%cd /content/repo
!grep -vE '^(torch|torchvision)==' requirements.txt > /tmp/req.txt && pip install -q -r /tmp/req.txt

## Data
First time only: download RUGD to Drive (~5 GB). Each session afterwards, copy it to local disk, which is much faster than reading from Drive during training.

In [ ]:
import os
if not os.path.exists(f'{PROJECT}/data/rugd/RUGD_annotations'):
    !bash scripts/download_rugd.sh {PROJECT}/data/rugd
!mkdir -p /content/data && rsync -a --exclude='*.zip' {PROJECT}/data/rugd /content/data/
os.environ['DATA_ROOT'] = '/content/data/rugd'
!python -m scripts.check_splits --root /content/data/rugd

## Train
Pick a config. `train.out_dir` points at Drive, so checkpoints survive disconnects.

In [ ]:
CONFIG = 'configs/e0_overfit.yaml'
OUT = f'{PROJECT}/runs'
# Optional: !wandb login, then add wandb.enabled=true below
!python -m src.train --config $CONFIG --set train.out_dir=$OUT

## Evaluate (val)
Select models on **val** only. Run the test split once per final config, with `--split test --confirm-test`.

In [ ]:
from pathlib import Path
RUN = f'{OUT}/' + Path(CONFIG).stem
!python -m src.evaluate --run $RUN --split val --out {PROJECT}/results